<a href="https://colab.research.google.com/github/safaabuzaid/mri-generalization/blob/main/03_EfficientNetB3_Augmentation.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# EfficientNet-B3 Augmentation Experiment

## Objective

In the previous experiment, ResNet18, DenseNet121, and EfficientNet-B3 were compared using three random seeds to evaluate their cross-domain generalization from Dataset A to the external Dataset B.

EfficientNet-B3 achieved the highest mean external test accuracy and showed the lowest variability across the three seeds. Therefore, it was selected for the next experiment.

The objective of this experiment is to investigate whether **data augmentation can further improve the cross-domain generalization of EfficientNet-B3**.

The augmented model will be compared with the baseline EfficientNet-B3 using the same three random seeds.

## Augmentation Pipeline

The following augmentations are applied **only to the training images**:

* Resize to 300 × 300 pixels
* Random horizontal flip
* Random rotation within ±10°
* Random affine transformation with small translation and scale variations
* Mild brightness and contrast variation
* Conversion to tensor
* ImageNet normalization

Validation and external test images will use the baseline preprocessing without augmentation.

## Evaluation

The augmented EfficientNet-B3 model will be trained using the same three random seeds:

* 42
* 123
* 2026

Performance will be evaluated on the external Dataset B using:

* Test accuracy
* Macro precision
* Macro recall
* Macro F1-score
* Per-class precision, recall, and F1-score
* Confusion matrix

The results will be compared with the baseline EfficientNet-B3 results to determine whether augmentation improves **external-domain generalization and stability across random seeds**.


In [ ]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [ ]:
import sys
src_path = '/content/drive/MyDrive/MRI_Generalization/src'

sys.path.append(src_path)

In [ ]:
import torch
import os
import random
import numpy as np
import pandas as pd
from data import BrainTumorDataset
from data_preparation import split_dataset, create_dataloaders, create_dataset
from train import train_model
from evaluate import test, create_confusion_matrix, classification_report
from transform import baseline_transform, augmentation_transform
from models import get_effecientnet_b3
from torch.utils.data import random_split


In [ ]:
#Reproducibility

def set_seed(seed):
    random.seed(seed)
    np.random.seed(seed)

    torch.manual_seed(seed)

    if torch.cuda.is_available():
        torch.cuda.manual_seed(seed)
        torch.cuda.manual_seed_all(seed)

    torch.backends.cudnn.deterministic = True
    torch.backends.cudnn.benchmark = False

SEEDS = [42, 123, 2026]

device = torch.device(
    "cuda" if torch.cuda.is_available() else "cpu"
)

print(device)

cuda


In [ ]:
SPLIT_DIR = "/content/drive/MyDrive/MRI_Generalization/splits"

train_df = pd.read_csv("/content/drive/MyDrive/MRI_Generalization/project_data/splits/train_split.csv")
val_df = pd.read_csv("/content/drive/MyDrive/MRI_Generalization/project_data/splits/val_split.csv")
test_df = pd.read_csv("/content/drive/MyDrive/MRI_Generalization/project_data/splits/test_split.csv")

print("Train:", len(train_df))
print("Validation:", len(val_df))
print("External Test:", len(test_df))

Train: 2451
Validation: 613
External Test: 2543


In [ ]:
# create datasets

train_dataset = BrainTumorDataset(
    dataframe=train_df,
    transform=augmentation_transform
)

val_dataset = BrainTumorDataset(
    dataframe=val_df,
    transform=baseline_transform
)

test_dataset = BrainTumorDataset(
    dataframe=test_df,
    transform=baseline_transform
)

In [ ]:
train_loader, val_loader, test_loader = create_dataloaders(train_dataset, val_dataset, test_dataset)

In [ ]:
from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    classification_report,
    confusion_matrix
)

In [ ]:
results = []
model_name = "efficientnet_b3_augmented"
NUM_EPOCHS = 10

for seed in SEEDS:

      print("=" * 60)
      print("Model: EfficientNet-B3")
      print(f"Seed: {seed}")
      print("=" * 60)

      set_seed(seed)

      model, criterion, optimizer, device = get_effecientnet_b3()

      # Train
      (
      history,
      best_epoch,
      best_train_loss,
      best_train_acc,
      best_val_loss,
      best_val_acc
      ) = train_model(
          model=model,
          train_loader=train_loader,
          val_loader=val_loader,
          optimizer=optimizer,
          criterion=criterion,
          device=device,
          num_epochs=NUM_EPOCHS
      )

      # Evaluate external dataset
      all_labels, all_preds = test(
          model,
          test_loader,
          device
      )

      test_accuracy = accuracy_score(all_labels, all_preds)

      macro_precision = precision_score(
          all_labels,
          all_preds,
          average="macro"
      )

      macro_recall = recall_score(
          all_labels,
          all_preds,
          average="macro"
      )

      macro_f1 = f1_score(
          all_labels,
          all_preds,
          average="macro"
      )

      print(f"Test Accuracy: {test_accuracy:.4f}")
      print(f"Macro Precision: {macro_precision:.4f}")
      print(f"Macro Recall: {macro_recall:.4f}")
      print(f"Macro F1: {macro_f1:.4f}")

      results.append({
          "model": "EfficientNetB3_aug",
          "seed": seed,

          "best_epoch": best_epoch,

          "train_loss": best_train_loss,
          "train_acc": best_train_acc,

          "val_loss": best_val_loss,
          "val_acc": best_val_acc,

          "test_accuracy": test_accuracy,
          "macro_precision": macro_precision,
          "macro_recall": macro_recall,
          "macro_f1": macro_f1
      })

      # confusion matrices
      cm = confusion_matrix(all_labels, all_preds)

      cm_df = pd.DataFrame(
          cm,
          index=["Glioma", "Meningioma", "Pituitary"],
          columns=["Glioma", "Meningioma", "Pituitary"]
      )

      cm_df.to_csv(
          f"cm_EfficientNetB3_aug_{seed}.csv"
      )

      #classification reports

      report = classification_report(
          all_labels,
          all_preds,
          target_names=[
              "Glioma",
              "Meningioma",
              "Pituitary"
          ],
          output_dict=True
      )

      report_df = pd.DataFrame(report).transpose()

      report_df.to_csv(
            f"classification_report_EfficientNetB3_aug_{seed}.csv"
        )


      results_df = pd.DataFrame(results)

      results_df.to_csv(
        "efficientnet_b3_augmentation_results.csv",
        index=False
      )

      print("Results saved.")


Model: EfficientNet-B3
Seed: 42
Downloading: "https://download.pytorch.org/models/efficientnet_b3_rwightman-b3899882.pth" to /root/.cache/torch/hub/checkpoints/efficientnet_b3_rwightman-b3899882.pth


100%|██████████| 47.2M/47.2M [00:00<00:00, 138MB/s]


Best model saved!
Epoch [1/10]
Train Loss: 0.5929, Train Acc: 0.7593
Val Loss: 0.2964, Val Acc: 0.8956
------------------------------
Best model saved!
Epoch [2/10]
Train Loss: 0.2256, Train Acc: 0.9196
Val Loss: 0.1602, Val Acc: 0.9462
------------------------------
Best model saved!
Epoch [3/10]
Train Loss: 0.1220, Train Acc: 0.9608
Val Loss: 0.1147, Val Acc: 0.9608
------------------------------
Best model saved!
Epoch [4/10]
Train Loss: 0.0717, Train Acc: 0.9772
Val Loss: 0.0873, Val Acc: 0.9690
------------------------------
Best model saved!
Epoch [5/10]
Train Loss: 0.0688, Train Acc: 0.9784
Val Loss: 0.0617, Val Acc: 0.9788
------------------------------
Epoch [6/10]
Train Loss: 0.0369, Train Acc: 0.9882
Val Loss: 0.0483, Val Acc: 0.9772
------------------------------
Best model saved!
Epoch [7/10]
Train Loss: 0.0413, Train Acc: 0.9849
Val Loss: 0.0470, Val Acc: 0.9804
------------------------------
Epoch [8/10]
Train Loss: 0.0464, Train Acc: 0.9861
Val Loss: 0.0494, Val Acc: 0.

In [ ]:
summary = pd.DataFrame({
    "Metric": [
        "Test Accuracy",
        "Macro Precision",
        "Macro Recall",
        "Macro F1"
    ],
    "Mean": [
        results_df["test_accuracy"].mean(),
        results_df["macro_precision"].mean(),
        results_df["macro_recall"].mean(),
        results_df["macro_f1"].mean()
    ],
    "Std": [
        results_df["test_accuracy"].std(),
        results_df["macro_precision"].std(),
        results_df["macro_recall"].std(),
        results_df["macro_f1"].std()
    ]
})

summary

,Metric,Mean,Std
0,Test Accuracy,0.951501,0.004314
1,Macro Precision,0.952989,0.003650
2,Macro Recall,0.951502,0.004266
3,Macro F1,0.951316,0.004284
